# Lab 3 - Linear Regression (Forest Fires)

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("forestfires.csv")
df = pd.get_dummies(df, columns=["month", "day"], dtype=float)  # chuyen chu thanh so (one-hot)

y = df["area"].values
X = df.drop(columns="area").values.astype(float)
names = list(df.drop(columns="area").columns)

# chia train/test: 80% / 20%
np.random.seed(0)
idx = np.random.permutation(len(X))
cut = int(0.8 * len(X))
X_train, X_test = X[idx[:cut]], X[idx[cut:]]
y_train, y_test = y[idx[:cut]], y[idx[cut:]]
print(X_train.shape, X_test.shape)

(413, 29) (104, 29)


### Assignment 1 (3 scores):

- Use the Numpy library only to construct the Linear Regression model.
- Train and Evaluate that Linear Regression model on the [Forest Fires](https://archive.ics.uci.edu/static/public/162/forest+fires.zip) dataset.

In [2]:
def fit(X, y):
    ones = np.ones((len(X), 1))
    X1 = np.hstack([ones, X])          # them cot 1 cho beta0
    return np.linalg.pinv(X1.T @ X1) @ X1.T @ y

def predict(X, beta):
    ones = np.ones((len(X), 1))
    return np.hstack([ones, X]) @ beta

def mse(y, y_pred):
    return np.mean((y - y_pred) ** 2)

def rmse(y, y_pred):
    return np.sqrt(mse(y, y_pred))

def mae(y, y_pred):
    return np.mean(np.abs(y - y_pred))

def r2(y, y_pred):
    return 1 - np.sum((y - y_pred) ** 2) / np.sum((y - y.mean()) ** 2)

def report(name, y, y_pred):
    print(name, "- MSE: %.2f  RMSE: %.2f  MAE: %.2f  R2: %.3f" % (mse(y, y_pred), rmse(y, y_pred), mae(y, y_pred), r2(y, y_pred)))

beta = fit(X_train, y_train)
pred = predict(X_test, beta)
report("Linear Regression", y_test, pred)

Linear Regression - MSE: 1413.69  RMSE: 37.60  MAE: 20.98  R2: -0.209


**Nhận xét:** Mô hình OLS tự viết bằng NumPy chạy được, nhưng chất lượng kém: R² trên tập test ≈ -0.21 (âm nghĩa là dự đoán tệ hơn việc luôn đoán giá trị trung bình). Nguyên nhân chính là `area` rất lệch (nhiều giá trị 0, vài giá trị rất lớn) nên mô hình tuyến tính khó khớp. RMSE và MAE cho biết sai số cùng đơn vị với `area`.

#### Cải thiện 1: log-transform cho `area`
Train trên `log(1 + area)`, sau đó đổi dự đoán về thang gốc bằng `expm1` để so sánh công bằng.

In [3]:
beta_log = fit(X_train, np.log1p(y_train))
pred_log = np.expm1(predict(X_test, beta_log))   # doi ve thang goc

report("Khong log", y_test, pred)
report("Co log   ", y_test, pred_log)

Khong log - MSE: 1413.69  RMSE: 37.60  MAE: 20.98  R2: -0.209
Co log    - MSE: 1223.94  RMSE: 34.98  MAE: 10.01  R2: -0.047


**Nhận xét:** Log-transform cải thiện nhẹ: MSE giảm, R² tăng từ -0.21 lên -0.05. MAE giảm mạnh (≈ 21.0 xuống ≈ 10.0) còn RMSE giảm ít (≈ 37.6 xuống ≈ 35.0). RMSE bị chi phối bởi vài vụ cháy rất lớn nên khó cải thiện, còn MAE phản ánh đa số trường hợp (cháy nhỏ) mà mô hình log dự đoán tốt hơn. R² vẫn âm.

#### Cải thiện 2: Cross-validation (5-fold)
Tập test chỉ có 104 mẫu nên kết quả phụ thuộc nhiều vào cách chia. Chia dữ liệu thành 5 phần, lần lượt dùng mỗi phần làm tập test, rồi lấy trung bình.

In [4]:
k = 5
np.random.seed(0)
idx = np.random.permutation(len(X))
folds = np.array_split(idx, k)

rmse_raw, rmse_log = [], []
for i in range(k):
    test_i = folds[i]
    train_i = np.concatenate([folds[j] for j in range(k) if j != i])

    b1 = fit(X[train_i], y[train_i])
    rmse_raw.append(rmse(y[test_i], predict(X[test_i], b1)))

    b2 = fit(X[train_i], np.log1p(y[train_i]))
    rmse_log.append(rmse(y[test_i], np.expm1(predict(X[test_i], b2))))

print("RMSE tung fold (khong log):", np.round(rmse_raw, 2))
print("RMSE tung fold (co log):   ", np.round(rmse_log, 2))
print("Trung binh khong log:", np.mean(rmse_raw).round(2), "+/-", np.std(rmse_raw).round(2))
print("Trung binh co log:   ", np.mean(rmse_log).round(2), "+/-", np.std(rmse_log).round(2))

RMSE tung fold (khong log): [109.3   24.63  75.27  33.59  37.68]
RMSE tung fold (co log):    [111.65  16.66  76.45  29.83  35.15]
Trung binh khong log: 56.1 +/- 31.73
Trung binh co log:    53.95 +/- 35.1


**Nhận xét:** RMSE giữa các fold chênh nhau rất lớn (từ ≈ 17 đến ≈ 111), kết quả phụ thuộc nhiều vào việc vài vụ cháy lớn rơi vào fold nào. Trung bình RMSE có log (≈ 54.0) chỉ nhỉnh hơn không log (≈ 56.1) trong khi độ lệch chuẩn cũng lớn, và có 2/5 fold log còn tệ hơn. Vậy lợi ích của log về RMSE chưa chắc chắn; lợi ích về MAE rõ hơn.

### Assignment 2 (3 scores):

- Standardize the data so that their mean is $0$ and their variance is $1$.
- Compare the results of Linear Regression model when being trained on the original and standardized data.

In [5]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1                      # tranh chia cho 0

X_train_s = (X_train - mean) / std
X_test_s = (X_test - mean) / std

beta_s = fit(X_train_s, y_train)
pred_s = predict(X_test_s, beta_s)

report("Du lieu goc ", y_test, pred)
report("Du lieu chuan hoa ", y_test, pred_s)

Du lieu goc        - MSE: 1413.69  RMSE: 37.60  MAE: 20.98  R2: -0.209
Du lieu chuan hoa  - MSE: 1413.69  RMSE: 37.60  MAE: 20.98  R2: -0.209


**Nhận xét:** Kết quả trên dữ liệu gốc và dữ liệu chuẩn hoá gần như giống hệt . Lý do: OLS có nghiệm đóng, đổi thang đo của cột chỉ làm đổi hệ số β tương ứng, còn dự đoán không đổi.

### Assignment 3 (2 scores):

- Perform feature engineering to remove columns that are linearly dependent (colinear) with other columns in the dataset.
- Evaluate the Linear Regression model when beeing trained on the original data and preprocessed data.

In [6]:
keep = []
for j in range(X_train.shape[1]):
    cols = keep + [j]
    A = np.hstack([np.ones((len(X_train), 1)), X_train[:, cols]])  # co ca cot 1
    if np.linalg.matrix_rank(A) == len(cols) + 1:
        keep = cols
    else:
        print("Bo cot:", names[j])

X_train_c = X_train[:, keep]
X_test_c = X_test[:, keep]

beta_c = fit(X_train_c, y_train)
pred_c = predict(X_test_c, beta_c)

report("Du lieu goc         ", y_test, pred)
report("Sau khi bo colinear ", y_test, pred_c)

Bo cot: month_sep
Bo cot: day_wed
Du lieu goc          - MSE: 1413.69  RMSE: 37.60  MAE: 20.98  R2: -0.209
Sau khi bo colinear  - MSE: 1413.69  RMSE: 37.60  MAE: 20.98  R2: -0.209


**Nhận xét:** Sau khi one-hot, các cột `month_*` cộng lại luôn bằng 1 (giống hệt cột hệ số chặn), `day_*` cũng vậy, nên mỗi nhóm có một cột thừa. Kiểm tra bằng `rank` thì bỏ được `month_sep` và `day_wed`. Kết quả đánh giá không đổi vì `pinv` vốn đã xử lý được ma trận suy biến; việc bỏ cột chủ yếu làm mô hình gọn và ổn định hơn một chút.

### Assignment 4 (2 scores):

- Implement the Linear Regression model using Machine Learning libraries (Scikit Learn or Skorch).
- Compare the results of Linear Regression model constructed manually and from Machine Learning libraries.

In [7]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)
pred_sk = model.predict(X_test)

report("Tu viet", y_test, pred)
report("Scikit-learn", y_test, pred_sk)
print("Du doan giong nhau?", np.allclose(pred, pred_sk))

Tu viet      - MSE: 1413.69  RMSE: 37.60  MAE: 20.98  R2: -0.209
Scikit-learn - MSE: 1413.69  RMSE: 37.60  MAE: 20.98  R2: -0.209
Du doan giong nhau? True


**Nhận xét:** Mô hình tự viết và `LinearRegression` của scikit-learn cho dự đoán trùng khớp (`np.allclose` = True, các chỉ số giống nhau)